In [1]:
import os

os.makedirs(".github/workflows", exist_ok=True)
print("Created .github/workflows directory.")

Created .github/workflows directory.


In [2]:
# Write main.yml file
yaml_content = """name: CI/CD Pipeline

on:
  push:
    branches: [ main ]
  pull_request:
    branches: [ main ]

jobs:
  lint:
    name: Lint Code Quality
    runs-on: ubuntu-latest
    steps:
      - name: Checkout Code
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.10'

      - name: Install Linting Tools
        run: |
          python -m pip install --upgrade pip
          pip install flake8

      - name: Run Syntax Checks
        run: |
          # Fails only on critical syntax errors or undefined names in main.py
          flake8 main.py --count --select=E9,F63,F7,F82 --show-source --statistics
          # Displays style warnings without failing the job
          flake8 main.py --count --exit-zero --max-line-length=88 --statistics

  test-and-dvc:
    name: DVC Tracking & Script Execution
    needs: lint
    runs-on: ubuntu-latest
    steps:
      - name: Checkout Code
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.10'

      - name: Install Dependencies
        run: |
          python -m pip install --upgrade pip
          pip install pytest dvc
          if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

      - name: Verify DVC Pointer Files
        run: |
          # Checks tracking status of dataset.csv.dvc & dataset_clean.csv.dvc
          dvc status || true

      - name: Execute Main Script
        run: |
          python main.py

  build:
    name: Build Docker Container
    needs: test-and-dvc
    runs-on: ubuntu-latest
    steps:
      - name: Checkout Code
        uses: actions/checkout@v4

      - name: Build Docker Image
        run: |
          docker build -t ml-pipeline:latest .
"""

with open(".github/workflows/main.yml", "w") as f:
    f.write(yaml_content)

print("Created .github/workflows/main.yml successfully.")

Writing .github/workflows/main.yml


In [3]:
# Stage the GitHub Actions workflow definition
!git add .github/workflows/main.yml

In [4]:
# Commit workflow configuration
!git commit -m "CI: Add GitHub Actions pipeline for main.py, DVC, and Docker build"

[main 97f3566] CI: Add GitHub Actions pipeline for main.py, DVC, and Docker build
 1 file changed, 51 insertions(+)
 create mode 100644 .github/workflows/main.yml


In [5]:
# Push changes to GitHub (assumes remote 'origin' is set)
!git push origin main

To https://github.com/ShreehariNair/insurance-data-science.git
   7cc1c7a..97f3566  main -> main
